# GTA 6 AutoClipper — browser-only practice session

Use this notebook in Google Colab so you do **not** need to install Linux, Python, or FFmpeg on your Chromebook.

## The fan-clipping rules this workflow follows

1. Work after the stream has ended. This notebook processes a file, not a live watch page.
2. Use public, non-subscriber-only footage, or footage you own. For another creator's video, get permission or a licence that covers editing and reposting to YouTube.
3. Prefer gameplay with cleared audio. If you cannot clear background music, you can mute the entire source audio; that also removes voice chat and game sound.
4. The output package includes source attribution text when you provide the source URL. **Credit is not permission**, and edits/captions do not guarantee that a copyright claim won't occur.
5. Review each candidate. Only selected candidates are included in the approved export ZIP. This notebook does not publish automatically.

**Important:** Kick's current Terms of Service restrict automated scraping and access through methods it hasn't provided or authorized. Download your own/authorized clip through an available platform feature or get the file from the creator; do not paste a hidden playback playlist URL. For your own YouTube videos, use YouTube Studio/Google Takeout to obtain the file.

Colab is a free, temporary processing session—not a 24/7 stream-monitoring server. Save the output ZIP before the runtime ends.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/gta6-autoclipper")
if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/topboyasian-stack/gta6-autoclipper.git", str(ROOT)],
        check=True,
    )
os.chdir(ROOT)

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.115,<1", "python-multipart>=0.0.9"],
    check=True,
)

print("Setup ready. The media tools run in Google's cloud runtime, not on your Chromebook.")

## Step 2 — choose source and generate candidates

For a clip from Kick, use the platform's available download option or obtain the file from the creator first. For your own YouTube uploads, use Studio or Takeout. For a large file, choose the Google Drive option to avoid uploading it twice.

In [ ]:
import json
import re
import shutil
import time
import uuid
import zipfile
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Video, Markdown
from google.colab import files

clip_seconds = int(input("Target clip length in seconds (10–90; recommend 30–40): ").strip() or "35")
max_candidates = int(input("Maximum candidates to generate (1–6; recommend 3): ").strip() or "3")
creator = input("Original creator/channel name: ").strip()
source_url = input("Original public source URL (optional but recommended): ").strip()
rights_basis = input("Rights basis (your own footage or the creator's permission/licence for editing and YouTube reposting): ").strip()

if len(rights_basis) < 8:
    raise ValueError("Enter the actual rights basis before processing.")
if input("Has the stream ended, and is the source public (not paid/subscriber/member-only) or your own footage? Type YES: ").strip() != "YES":
    raise RuntimeError("Stopped. Only continue with a completed public source or footage you own.")
if input("Do you own this footage or have permission/licence covering editing AND YouTube reposting? Type YES: ").strip() != "YES":
    raise RuntimeError("Stopped. Public availability and attribution alone do not grant reposting rights.")

mute_audio = input("Mute all source audio if background music is not cleared? (y/N): ").strip().lower() == "y"
os.environ["CLIP_SECONDS"] = str(max(10, min(90, clip_seconds)))
os.environ["MAX_CANDIDATES"] = str(max(1, min(6, max_candidates)))
os.environ["MUTE_SOURCE_AUDIO"] = "1" if mute_audio else "0"

import app.main as clipper
clipper.CLIP_SECONDS = max(10, min(90, clip_seconds))
clipper.MAX_CANDIDATES = max(1, min(6, max_candidates))
DATA, UPLOADS, CLIPS, db, create_candidates = clipper.DATA, clipper.UPLOADS, clipper.CLIPS, clipper.db, clipper.create_candidates

choice = input("Source file: type 1 to upload from this device, or 2 to choose a file already in Google Drive: ").strip()
if choice == "2":
    from google.colab import drive
    drive.mount("/content/drive")
    source_file = Path(input("Paste the full Drive file path, e.g. /content/drive/MyDrive/clips/my_clip.mp4: ").strip())
    if not source_file.is_file():
        raise FileNotFoundError(f"File not found: {source_file}")
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    if len(uploaded) > 1:
        print("More than one file selected; using the first file.")
    uploaded_name = next(iter(uploaded))
    source_file = Path(uploaded_name)

allowed = {".mp4", ".mov", ".mkv", ".webm", ".m4v"}
suffix = source_file.suffix.lower()
if suffix not in allowed:
    raise ValueError(f"Unsupported video file type {suffix!r}. Use MP4, MOV, MKV, WebM or M4V.")

source_id = uuid.uuid4().hex
safe_name = re.sub(r"[^A-Za-z0-9._-]", "_", source_file.name)[:100]
saved_source = UPLOADS / f"{source_id}_{safe_name}"
if source_file.resolve() != saved_source.resolve():
    shutil.copy2(source_file, saved_source)

with db() as con:
    con.execute(
        "INSERT INTO sources(id,filename,path,creator,source_url,rights_basis) VALUES(?,?,?,?,?,?)",
        (source_id, safe_name, str(saved_source), creator, source_url, rights_basis),
    )

result = create_candidates(source_id, saved_source)
with db() as con:
    rows = con.execute(
        "SELECT c.*, s.filename, s.creator, s.source_url, s.rights_basis FROM clips c JOIN sources s ON s.id=c.source_id "
        "WHERE c.source_id=? ORDER BY c.start",
        (source_id,),
    ).fetchall()

candidate_records = []
approval_checks = []
display(Markdown(f"## {len(rows)} candidate(s) generated from {result['filename']}"))
display(Markdown("Watch each vertical preview. Tick only clips you approve for export. The unticked clips will be rejected when you apply the decision."))
for index, row in enumerate(rows, 1):
    item = dict(row)
    candidate_records.append(item)
    display(Markdown(f"### Candidate {index} — start around {int(row['start'])}s"))
    display(Video(filename=str(row["path"]), embed=True, width=320))
    checkbox = widgets.Checkbox(value=False, description=f"Approve candidate {index} for export")
    approval_checks.append(checkbox)
    display(checkbox)

display(Markdown("Next, run the 'Apply review and export approved clips' cell below."))

## Step 3 — review and export approved clips

Use the checkboxes shown under each preview, then run the next cell. Only checked clips are exported.

In [ ]:
if not candidate_records:
    raise RuntimeError("No candidates exist. Run the processing cell first.")

approved = []
with db() as con:
    for item, checkbox in zip(candidate_records, approval_checks):
        decision = "approved" if checkbox.value else "rejected"
        con.execute("UPDATE clips SET status=? WHERE id=? AND status='pending'", (decision, item["id"]))
        if checkbox.value:
            approved.append(item)

if not approved:
    print("No clips were approved, so no export ZIP was created.")
else:
    stamp = time.strftime("%Y%m%d_%H%M%S")
    export_dir = Path("/content/autoclipper-approved")
    export_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path(f"/content/gta6-autoclipper-approved-{stamp}.zip")

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        manifest = []
        for index, item in enumerate(approved, 1):
            clip_path = Path(item["path"])
            base = re.sub(r"[^A-Za-z0-9._-]", "_", Path(item["filename"]).stem)[:60] or "gaming_clip"
            out_name = f"{index:02d}_{base}_{int(item['start'])}s.mp4"
            archive.write(clip_path, arcname=out_name)

            title = f"{base} highlight #Shorts"
            description_lines = [
                "Fan edit / gaming highlight.",
                "Original creator: " + (item.get("creator") or creator or "Not supplied"),
            ]
            original_url = (item.get("source_url") or "").strip()
            if original_url.startswith(("https://", "http://")):
                description_lines.append("Original source: " + original_url)
            description_lines += [
                "",
                "This video has been reframed and cut from the supplied source footage.",
                "Please confirm rights to all video and audio before publishing. Attribution is not a substitute for permission."
            ]
            metadata_name = out_name.removesuffix(".mp4") + "_youtube_metadata.txt"
            metadata_text = "Suggested title: " + title + "\n\nSuggested description:\n" + "\n".join(description_lines) + "\n"
            archive.writestr(metadata_name, metadata_text)
            manifest.append({
                "file": out_name, "clip_id": item["id"], "start_seconds": item["start"],
                "duration_seconds": item["duration"], "creator": item.get("creator"),
                "source_url": original_url, "rights_basis_recorded": bool(item.get("rights_basis"))
            })
        archive.writestr("approval_manifest.json", json.dumps(manifest, indent=2))

    print(f"Export ready: {len(approved)} approved clip(s).")
    print("The ZIP contains MP4s, a suggested title/description file for each, and an approval manifest.")
    print("Next: upload each MP4 in YouTube Studio as PRIVATE, check the private copy, then publish manually if rights and audio are cleared.")
    files.download(str(zip_path))

## Publishing after review

The ZIP contains only clips you explicitly approved plus a suggested title/description file with source attribution when available. In YouTube Studio, upload each MP4 as **Private** first. Check the private version, confirm the video/audio rights, then change visibility yourself. Attribution reduces confusion, but it does not grant rights or guarantee the absence of Content ID claims. YouTube's monetization policy also reviews reused content and expects meaningful original value; merely adding zooms, captions, or pacing cuts may not make a channel eligible for monetization.

- [Kick Terms of Service](https://kick.com/terms-of-service)
- [YouTube channel monetization policies](https://support.google.com/youtube/answer/1311392)
- [YouTube Data API upload guide](https://developers.google.com/youtube/data/guides/uploading_a_video)